# Inter-slide generalization of the frozen ASTER WBC localizer

## tl;dr

The historical 960-pixel model that produced **mAP@50 = 0.9823** was frozen and evaluated without retraining or threshold adjustment on two additional slides. The slides contain **159 fields and 241 annotated WBCs** in total.

- Unseen slide 25/09: mAP@50 **0.7749**, precision **0.8652**, recall **0.6063** at confidence 0.18.
- Unseen slide 26/09: mAP@50 **0.9939**, precision **0.9908**, recall **0.9474** at confidence 0.18.
- Pooled external slides: mAP@50 **0.9038**, precision **0.9343**, recall **0.7676** at confidence 0.18.

These results support **preliminary inter-slide generalization**, while the difference between the two new slides shows that performance is not uniform across slides.

## Context & Methods

### Key assumptions

- Each top-level folder under `captures_Lame/` represents a different slide.
- The new slides were never used for training, validation, model selection, or threshold selection.
- All annotations are single-class YOLO labels (`0 = WBC`).
- The frozen checkpoint must have SHA-256 `5253be6b…39a1`, the exact historical model evaluated at 0.9823.

The standard Ultralytics `DetectionValidator` is used at `imgsz=960`, class 0, NMS IoU 0.50, and `max_det=300`. AP uses a confidence floor of 0.001. TP/FP/FN use the prespecified deployment confidence of 0.18 and TP IoU ≥ 0.50.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path(os.environ.get("ASTER_REPO_ROOT", Path.cwd())).resolve()
EXPERIMENT = ROOT / "experiments/inter_slide_generalization"
RESULTS = EXPERIMENT / "results"
RUN_FULL_EVALUATION = False  # Set True after placing the two slide folders under external_data/unseen_slides/.

assert (EXPERIMENT / "run_inter_slide_evaluation.py").is_file()
assert RESULTS.is_dir()
print("Experiment:", EXPERIMENT)
print("Full evaluation requested:", RUN_FULL_EVALUATION)


### 1. Optional full rerun

Leave `RUN_FULL_EVALUATION = False` to inspect the verified saved results. Set it to `True` to validate the data, verify the model checksum, and repeat all six standard validation passes on Apple MPS.

In [ ]:
if RUN_FULL_EVALUATION:
    command = [
        sys.executable,
        str(EXPERIMENT / "run_inter_slide_evaluation.py"),
        "--device", "mps",
        "--batch", "8",
    ]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Using saved, already executed results.")


## Data

The next cell checks the recorded protocol and displays the standard metrics. Precision, recall, and F1 suffixed `conf018` correspond to the fixed deployment operating point.

In [ ]:
metrics = pd.read_csv(RESULTS / 'inter_slide_metrics.csv')
provenance = json.loads((RESULTS / 'provenance.json').read_text())

assert provenance['protocol']['training'] is False
assert provenance['protocol']['threshold_tuning_on_external_slides'] is False
assert provenance['protocol']['weights_sha256'] == '5253be6be234b956fa0da9af575949b155b9e47917fb79301d5cb62f8a2d39a1'
assert metrics['reference_wbc'].sum() >= 241  # includes the pooled row

shown = metrics[[
    'dataset', 'images', 'reference_wbc', 'map50', 'map50_95',
    'tp', 'fp', 'fn', 'precision_conf018', 'recall_conf018', 'f1_conf018'
]].copy()
for column in ['map50', 'map50_95', 'precision_conf018', 'recall_conf018', 'f1_conf018']:
    shown[column] = shown[column].round(4)
display(shown)

## Results

The original held-out fields from the adaptation slide are included only as a historical comparator. The two dated groups are entirely unseen slides.

In [ ]:
external = metrics[metrics.dataset != 'combined_unseen_slides'].copy()
historical = pd.DataFrame([{
    'dataset': 'Original within-slide\n(same slide)',
    'map50': 0.9823130169433179,
    'precision_conf018': 164 / 183,
    'recall_conf018': 164 / 171,
}])
external['dataset'] = external['dataset'].str.replace('captures_Lame', 'Unseen slide\n', regex=False)
plot_data = pd.concat([historical, external], ignore_index=True)

fig, ax = plt.subplots(figsize=(9.2, 5.2))
plot_data.set_index('dataset')[[
    'map50', 'precision_conf018', 'recall_conf018'
]].rename(columns={
    'map50': 'mAP@50',
    'precision_conf018': 'Precision (conf=0.18)',
    'recall_conf018': 'Recall (conf=0.18)',
}).plot.bar(ax=ax, color=['#2F6B9A', '#3C9D77', '#D67B32'], width=0.72)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_xlabel('')
ax.set_title('Frozen localizer: within-slide baseline and unseen-slide transfer')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False, loc='lower center', ncol=3)
ax.tick_params(axis='x', rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

The second unseen slide closely reproduces the original within-slide performance. The first unseen slide is substantially harder, particularly at the frozen deployment threshold, where 50 of 127 reference WBCs are missed. Pooling both new slides gives mAP@50 0.9038, with 185 TP, 13 FP, and 56 FN at confidence 0.18.

## Manuscript-ready wording

> To assess whether the high target-domain performance reflected slide-specific adaptation, the historical 960-pixel localizer was frozen and evaluated without retraining, model selection, or threshold adjustment on two additional educational blood-smear slides acquired independently with the same ASTER ×40 setup. The external set comprised 159 fields and 241 annotated WBCs. Performance varied between slides: mAP@50 was 0.775 on the first unseen slide and 0.994 on the second, compared with 0.982 on the original within-slide held-out test. Pooled across the two unseen slides, mAP@50 was 0.904 and mAP@50–95 was 0.606. At the prespecified deployment operating point (confidence 0.18; NMS IoU 0.50), pooled precision was 0.934 and recall was 0.768 (185 TP, 13 FP, and 56 FN). These results provide preliminary evidence of inter-slide transfer while also showing appreciable slide-to-slide variability.

### Limitation wording

> The inter-slide evaluation included only two additional slides, both acquired with the same device and annotated by one operator. It therefore does not establish robustness across laboratories, staining protocols, acquisition devices, or annotators. Independent annotation review and evaluation on more slides remain necessary.

## Takeaways

1. The critique is now tested directly: neither new slide contributed to adaptation or threshold selection.
2. The pooled external result is strong enough to report, but the 25/09 slide prevents claiming uniformly preserved performance.
3. Report each slide separately; reporting only the pooled mAP would conceal the observed variability.
4. Two new slides improve the evidence, while a third to fifth slide and independent annotation review would further strengthen the claim.